# 08 — Pigeon DMD: Dynamic Mode Decomposition for pigeon flight

This notebook shows you how to use **BirdDMD** with pigeon data.

**What is DMD?** Dynamic Mode Decomposition decomposes a time series of measurements (like 3D wing marker positions during flight) into a small set of **oscillatory modes**. Each mode has:
- a **frequency** (how fast it oscillates — e.g. the wingbeat frequency)
- a **spatial pattern** (which markers move, and in what direction)
- an **amplitude** (how strong the mode is)

For pigeon flight, we typically find:
- A **wingbeat mode** (~7 Hz) — the primary up-down flapping stroke
- A **doubled-frequency mode** (~14 Hz) — captures the asymmetry between upstroke and downstroke
- A **base mode** (0 Hz) — the mean wing posture everything oscillates around

**About this notebook:** We don't have real pigeon DMD data to hand yet, so we generate **synthetic** flight data by adding realistic sinusoidal wingbeat motion to the mean pigeon shape. This lets us demonstrate the full workflow. When you have real data, you just swap out the synthetic data generation step — everything else is identical.

> **If you've used the PCA tutorial**, you'll recognise the `Animal3D` pigeon object. We use it here too, but instead of asking "what shapes does the wing take?" we ask "how does the wing *move over time*?".

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import matplotlib.pyplot as plt
import numpy as np

# morphing_birds: tools for the 3D pigeon skeleton and visualisation
from morphing_birds import (
    Animal3D,
    animate_plotly_compare,
    plot_plotly,
    plot_plotly_with_trace,
)

# birddmd: the DMD analysis tools
from birddmd import (
    compute_rmse,
    convergence_analysis,
    normalise_data,
    plot_amplitude_ranking,
    plot_convergence,
    plot_mode_dynamics,
    reconstruct,
    run_dmd,
    variance_explained,
)

## 1 — Load the pigeon skeleton

We start by creating an `Animal3D` object for the pigeon. If you've used the PCA tutorial, this will look familiar.

The `variant='simple'` argument selects a reduced set of 8 markers that directly matches the hawk skeleton used throughout the rest of BirdDMD:

| Marker | Left | Right |
|--------|------|-------|
| Wingtip | ✓ | ✓ |
| Wrist | ✓ | ✓ |
| Secondary | ✓ | ✓ |
| Tailtip | ✓ | ✓ |

Using 8 markers keeps the analysis consistent with the hawk notebooks and is the right choice when you want to compare across species.

In [ ]:
# Create the pigeon object — 'simple' variant gives us 8 analysis markers
pigeon = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv", variant="simple")
print(pigeon)

print(f"\nAnalysis markers ({len(pigeon.analysis_marker_names)}):")
for i, name in enumerate(pigeon.analysis_marker_names):
    print(f"  {i}: {name}")

In [ ]:
# Visualise the mean pigeon shape — you can rotate this 3D plot interactively
fig = plot_plotly(pigeon, colour="cornflowerblue")
fig.update_layout(title="Mean pigeon shape (simple variant, 8 markers)")
fig.show()

## 2 — Create synthetic pigeon flight data

We'll build a fake flight sequence by adding sinusoidal motion to the mean shape.

**Why synthetic?** It lets us demonstrate the full DMD workflow with data we can inspect and understand. When you have real pigeon motion capture files, you'd replace this section with:
```python
motion_data = pigeon.load_motion_data('your_file.csv')
synthetic_data = pigeon.get_analysis_data(motion_data)
```

**What are we simulating?**
- A pigeon flaps at roughly **7 Hz** — this is our primary wingbeat frequency
- The downstroke and upstroke are not mirror images of each other, which creates a **14 Hz harmonic** (twice the wingbeat)
- Both oscillations are mainly vertical (z-axis), with a smaller fore-aft component (y-axis)
- The wingtips move the most; the tail barely moves

In [ ]:
# --- Time parameters ---
WINGBEAT_FREQ = 7.0  # Hz — typical pigeon wingbeat
N_CYCLES = 2  # number of wingbeat cycles to simulate
N_FRAMES = 200  # time steps

duration = N_CYCLES / WINGBEAT_FREQ  # ~0.286 s
times = np.linspace(0, duration, N_FRAMES)
dt = times[1] - times[0]

print(f"Simulating {N_CYCLES} wingbeat cycles")
print(f"Duration:   {duration:.3f} s")
print(f"Time step:  {dt * 1000:.2f} ms")
print(f"Frames:     {N_FRAMES}")

In [ ]:
# --- Start from the mean shape ---
# pigeon.markers returns shape (1, n_markers, 3) — the leading 1 is the frame dimension
# We take [0] to get (n_markers, 3) for the mean shape alone
mean_shape = pigeon.markers[0].copy()  # (8, 3)
n_markers = mean_shape.shape[0]
print(f"Mean shape: {mean_shape.shape}  — {n_markers} markers x 3 coordinates")

# --- Define how much each marker oscillates vertically (z-axis) ---
# Wingtips move most; tail barely moves.
# We assign amplitudes by checking which marker it is.
z_amplitudes = np.zeros(n_markers)
for i, name in enumerate(pigeon.analysis_marker_names):
    if "wingtip" in name:
        z_amplitudes[i] = 0.40
    elif "wrist" in name:
        z_amplitudes[i] = 0.28
    elif "secondary" in name:
        z_amplitudes[i] = 0.18
    elif "tailtip" in name:
        z_amplitudes[i] = 0.08

# --- Build the time series ---
omega1 = 2 * np.pi * WINGBEAT_FREQ  # primary frequency (rad/s)
omega2 = 2 * np.pi * (2 * WINGBEAT_FREQ)  # doubled frequency (rad/s)

# Start with the mean shape repeated for every frame
synthetic_data = np.tile(mean_shape, (N_FRAMES, 1, 1))  # (200, 8, 3)

for i in range(n_markers):
    amp = z_amplitudes[i]
    # Primary wingbeat: sinusoidal in z
    synthetic_data[:, i, 2] += amp * np.sin(omega1 * times)
    # Doubled frequency: upstroke/downstroke asymmetry (30% amplitude)
    synthetic_data[:, i, 2] += 0.30 * amp * np.sin(omega2 * times)
    # Fore-aft motion in y (20% amplitude, 90 deg phase shifted)
    synthetic_data[:, i, 1] += 0.20 * amp * np.cos(omega1 * times)

# Small noise to help numerical stability in the DMD fit
rng = np.random.default_rng(42)
synthetic_data += rng.normal(0, 0.001, synthetic_data.shape)

print(f"\nSynthetic data shape: {synthetic_data.shape}")
print(
    f"  {synthetic_data.shape[0]} frames x "
    f"{synthetic_data.shape[1]} markers x "
    f"{synthetic_data.shape[2]} coordinates"
)

In [ ]:
# Sanity check: plot the left wingtip z-coordinate over time
# We expect a sinusoidal trace combining the 7 Hz and 14 Hz oscillations
wt_idx = pigeon.analysis_marker_names.index("left_wingtip")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times * 1000, synthetic_data[:, wt_idx, 2], color="cornflowerblue", lw=1.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("z coordinate")
ax.set_title("Synthetic wingbeat — left wingtip vertical motion")

# Mark the period of one wingbeat
period_ms = 1000 / WINGBEAT_FREQ
ax.axvline(
    period_ms,
    color="grey",
    ls="--",
    lw=0.8,
    label=f"One wingbeat period ({period_ms:.0f} ms)",
)
ax.legend()
plt.tight_layout()
plt.show()

z_vals = synthetic_data[:, wt_idx, 2]
print(f"Wingtip z range: {z_vals.min():.3f} to {z_vals.max():.3f}")

## 3 — Average shape for DMD centring

DMD works on **deviations from a mean**. Before fitting, it subtracts the average posture from every frame so it can focus on the oscillatory changes rather than the static position.

We provide the average shape explicitly. Since we built the synthetic data *from* the mean shape, `pigeon.markers` is exactly right here. With real flight data you'd compute the mean over your sequence instead.

In [ ]:
# pigeon.markers returns (1, n_markers, 3) — the mean shape loaded from CSV
average_shape = pigeon.markers  # shape (1, 8, 3)
print(f"Average shape: {average_shape.shape}")
print("  This is the static posture DMD will subtract before fitting.")

## 4 — Convergence analysis: how many modes do we need?

Before committing to a number of DMD modes, we do a quick sweep: fit DMD with 2, 4, 6, 8, ... modes and track how much reconstruction error drops.

**What to look for:** The error should drop sharply up to some number of modes, then plateau. The elbow point is where adding more modes stops being worth it.

For our synthetic data, we built exactly two frequencies plus a mean, so we expect the elbow at **6 modes** (= 3 conjugate pairs: wingbeat, doubled frequency, base).

In [ ]:
conv_results = convergence_analysis(
    markers=synthetic_data,
    times=times,
    average_shape=average_shape,
    n_markers=n_markers,
    max_modes=12,
    verbose=False,
)

fig = plot_convergence(conv_results)
plt.show()

print("\nSummary:")
for n, rmse, ve in zip(
    conv_results["n_modes"],
    conv_results["rmse_mean"],
    conv_results["variance_explained"],
    strict=False,
):
    print(f"  {n:2d} modes — RMSE: {rmse:.5f}, variance explained: {ve * 100:.1f}%")

## 5 — Run DMD with 6 modes

Now we fit the full DMD model.

**Key parameters:**
- `n_modes=6` — 6 modes = 3 conjugate pairs (chosen from the convergence analysis above)
- `d=2` — Hankel delay embedding of depth 2. This gives DMD a bit more context about how the data evolves in time.
- `eig_constraints={'conjugate_pairs'}` — forces eigenvalues to come in complex-conjugate pairs. This is physically correct for oscillatory signals: every oscillating mode has a positive-frequency and negative-frequency counterpart.
- `average_shape` — the mean posture to subtract before fitting (from step 3)

**What is a conjugate pair?** A complex number $a + bi$ and its conjugate $a - bi$ together describe a pure oscillation. The imaginary part $b$ tells you the frequency; the real part $a$ tells you whether the oscillation is growing or decaying.

In [ ]:
N_MODES = 6
D = 2

result = run_dmd(
    data=synthetic_data,
    times=times,
    n_modes=N_MODES,
    d=D,
    eig_constraints={"conjugate_pairs"},
    n_markers=n_markers,
    average_shape=average_shape,
    verbose=True,
)

print(f"\nDMD found {result.n_pairs} conjugate pairs:")
for idx in range(result.n_pairs):
    freq = result.pair_frequency(idx)
    i, j = result.conjugate_pairs[idx]
    print(f"  Pair {idx}: modes ({i},{j}), frequency = {freq:.2f} Hz")

In [ ]:
# Amplitude ranking: how much does each mode contribute?
# The taller the bar, the more that mode drives the wing motion.
fig, ax, _ = plot_amplitude_ranking(
    synthetic_data,
    times,
    max_modes=10,
    d=D,
    eig_constraints={"conjugate_pairs"},
    normalise_fn=lambda m: normalise_data(m, average_shape),
)
plt.show()

In [ ]:
# Mode dynamics: time traces for each mode
fig = plot_mode_dynamics(times[1:], result, axes_visible=False)
plt.show()

## 6 — Inspect the DMDResult

The `run_dmd()` function returns a `DMDResult` object — a container holding everything DMD computed. Think of it like a dictionary, but with dot-access and some built-in helper methods.

Here's a quick tour of what's inside:

In [ ]:
print("DMDResult fields:")
print(
    f"  eigenvalues     {result.eigenvalues.shape}"
    "  — complex numbers encoding frequency & growth"
)
print(f"  modes           {result.modes.shape}  — spatial pattern for each mode")
print(f"  amplitudes      {result.amplitudes.shape}  — strength of each mode")
print(f"  frequencies_hz  {result.frequencies_hz.shape}  — oscillation frequency (Hz)")
print(
    f"  growth_rates    {result.growth_rates.shape}  — growth/decay rate (0 = stable)"
)
print(f"  conjugate_pairs {result.conjugate_pairs}")
print(
    f"  reconstruction  {result.reconstruction.shape}  — full reconstructed time series"
)

print(f"\nFrequencies (|Hz|): {np.round(np.abs(result.frequencies_hz), 2)}")
print(f"Growth rates:       {np.round(result.growth_rates, 4)}")
print()
print("Growth rates near zero = stable oscillations (expected for periodic flapping).")

## 7 — Reconstruct individual modes

To understand what each mode *means physically*, we reconstruct the 3D marker trajectories from each conjugate pair in isolation.

The `reconstruct()` function takes:
- `result` — the DMDResult
- `times` — the time points to reconstruct at (note: DMD uses `times[1:]` due to the delay embedding)
- `pairs` — which conjugate pair(s) to include

We expect:
- **Wingbeat (~7 Hz)** — large vertical strokes of the wingtips
- **Doubled frequency (~14 Hz)** — smaller asymmetric shaping
- **Base (0 Hz)** — a static offset representing mean posture

In [ ]:
# Label each pair by its frequency.
# Expected frequencies: ~0 Hz (base), ~7 Hz (wingbeat), ~14 Hz (doubled).
# Any pair outside these ranges is labelled 'Other' — this can happen when BOPDMD
# picks up a numerical artefact mode from near-perfectly sinusoidal data.
NEAR_ZERO_HZ = 0.5

pair_labels = []
pair_colours = []
colour_map = {
    "Base": "#EE7447",
    "Wingbeat": "#DF5D99",
    "Doubled": "#57B7B0",
    "Other": "#AAAAAA",
}

for idx in range(result.n_pairs):
    freq = result.pair_frequency(idx)
    gr = result.growth_rates[result.conjugate_pairs[idx][0]]
    if freq < NEAR_ZERO_HZ:
        label = "Base"
    elif abs(freq - WINGBEAT_FREQ) < 1.0:
        label = "Wingbeat"
    elif abs(freq - 2 * WINGBEAT_FREQ) < 2.0:
        label = "Doubled"
    else:
        label = "Other"
    pair_labels.append(label)
    pair_colours.append(colour_map[label])
    print(f"  Pair {idx}: {label:10s}  {freq:.2f} Hz  (growth rate: {gr:.4f})")

print()
print("Note: an 'Other' mode with a large growth rate is a numerical artefact.")
print("It contributes little — check the amplitude ranking above.")

In [ ]:
# Reconstruct each pair's contribution to the wing motion
mode_keypoints = []
for pair_idx in range(result.n_pairs):
    kp = reconstruct(result, times=times[1:], pairs=[pair_idx])
    mode_keypoints.append(kp)
    print(f"Pair {pair_idx} ({pair_labels[pair_idx]}): {kp.shape}")

In [ ]:
# Visualise each mode as a 3D trace
# The trace shows the path each marker traces through space over one cycle
for pair_idx, (kp, colour, label) in enumerate(
    zip(mode_keypoints, pair_colours, pair_labels, strict=False)
):
    freq = result.pair_frequency(pair_idx)
    pigeon_viz = Animal3D(
        "pigeon", data="../data/mean_pigeon_shape.csv", variant="simple"
    )
    pigeon_viz.update_keypoints(kp[0])

    fig = plot_plotly_with_trace(pigeon_viz, keypoints_frames=kp, colour=colour)
    fig.update_layout(
        title=f"Mode {pair_idx}: {label} ({freq:.1f} Hz)",
        scene={"camera": {"eye": {"x": 0.0, "y": 1.2, "z": 0.4}}},
    )
    fig.show()

In [ ]:
# Animated side-by-side comparison of all three modes
pigeon_anim = Animal3D("pigeon", data="../data/mean_pigeon_shape.csv", variant="simple")
animate_plotly_compare(
    pigeon_anim, keypoints_frames_list=mode_keypoints, colours=pair_colours
)

## 8 — Full reconstruction and accuracy

Summing all three mode pairs should recover the original data. Let's check how accurate the reconstruction is.

Because of the Hankel delay embedding (`d=2`), the reconstruction covers `times[1:]` — one frame shorter than the input. This is expected behaviour.

For our synthetic data (built from exactly two frequencies plus a mean), we expect near-perfect reconstruction — well over 99% variance explained.

In [ ]:
recon = result.reconstruction  # (n_frames-1, 8, 3)
ground_truth = synthetic_data[1:]  # skip first frame to match

rmse = compute_rmse(recon, ground_truth)  # per-frame RMSE
ve = variance_explained(ground_truth, recon)

print(f"Reconstruction shape:  {recon.shape}")
print(f"Ground truth shape:    {ground_truth.shape}")
print(f"Mean RMSE:             {np.mean(rmse):.6f}")
print(f"Std RMSE:              {np.std(rmse):.6f}")
print(f"Variance explained:    {ve * 100:.2f}%")

In [ ]:
# RMSE over time — should be low and fairly flat
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times[1:] * 1000, rmse, color="steelblue", lw=1.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("RMSE")
ax.set_title("Per-frame reconstruction error")
ax.axhline(
    np.mean(rmse),
    color="tomato",
    ls="--",
    lw=1,
    label=f"Mean RMSE = {np.mean(rmse):.5f}",
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Animated comparison: reconstruction (red) vs original data (black)
pigeon_recon = Animal3D(
    "pigeon", data="../data/mean_pigeon_shape.csv", variant="simple"
)
animate_plotly_compare(
    pigeon_recon,
    keypoints_frames_list=[recon, ground_truth],
    colours=["red", "black"],
)

## 9 — Next steps: using your real data

### Swapping in real pigeon data

Replace the synthetic data generation (Section 2) with your own motion capture data:

```python
# Load your CSV (uses the column mapping defined in the pigeon config)
motion_data = pigeon.load_motion_data('your_pigeon_file.csv')

# Slice to the 8 analysis markers
synthetic_data = pigeon.get_analysis_data(motion_data)   # (n_frames, 8, 3)

# times: create from your frame rate, e.g. 200 Hz
times = np.arange(synthetic_data.shape[0]) / 200.0
```

If you have **multiple flight sequences**, you'll want to bin and average them first (like notebook 01 does for the hawk). Check `notebooks/scripts/prepare_flapping.py` for the hawk pipeline — the pigeon equivalent would follow the same pattern.

### Changing the number of markers

The `variant='simple'` gives 8 markers for cross-species compatibility. If you want to use the full 14-marker pigeon instead:

```python
pigeon_full = Animal3D('pigeon', data='../data/mean_pigeon_shape.csv')  # no variant
n_markers   = len(pigeon_full.analysis_marker_names)                    # 14

result = run_dmd(..., n_markers=n_markers, ...)
```

### Going deeper

Once you're comfortable with this workflow, explore the other notebooks:

| Notebook | Topic |
|----------|-------|
| `03_double_frequency.ipynb` | Deep dive into the 14 Hz asymmetry mode |
| `06_reconstruction_accuracy.ipynb` | Batch RMSE across many sequences |
| `07_generative_model.ipynb` | Using DMD to generate novel flight sequences |

These are all shown for the hawk, but the approach transfers directly to pigeon data.